# Comparación del código de control de robots seriales



# Documentación del Nodo: `scara_tray_line_py.py`

Este script inicializa un nodo de **ROS 2** en Python diseñado para controlar las trayectorias de un robot tipo SCARA en línea recta.

## Descripción del Bloque

Este código implementa un nodo de control de trayectoria en lazo abierto para un robot manipulador tipo SCARA de 3 grados de libertad utilizando el entorno de desarrollo ROS 2 (Robot Operating System) en Python.

Su objetivo principal es calcular y publicar los comandos de movimiento articular necesarios para que el extremo del robot (efector final) se desplace siguiendo una trayectoria rectilínea perfecta en el espacio cartesiano bidimensional.


**#!/usr/bin/env python3** #<----- Shebang: Indica al sistema operativo que ejecute el archivo con Python 3.

### Importación de librerías de ROS 2 y tipos de mensajes
**import rclpy** #<----- Importa la librería cliente principal de ROS 2 para Python.

**from rclpy.node import Node** #<----- Importa la clase base Node para crear y gestionar nodos en ROS 2.

### Interfaces de mensajes de ROS 2 para trayectorias de articulaciones y duraciones
**from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint** #<----- Importa los tipos de mensajes necesarios para enviar comandos de trayectorias multieje a las articulaciones.

**from builtin_interfaces.msg import Duration** #<----- Importa el tipo de mensaje para definir duraciones de tiempo específicas de ROS 2.

### Importación de librerías matemáticas y de tiempo
**import time** #<----- Importa el módulo estándar de Python para gestionar funciones de tiempo.

**from math import cos, sin, acos, asin, atan2, sqrt** #<----- Importa funciones trigonométricas y matemáticas para la cinemática del robot.


### Definición de la clase del nodo para el robot SCARA
**class ScaraTrayLineNode(Node):** #<----- Declara la clase del nodo que hereda de la clase base Node.

**def __init__(self):** #<----- Método constructor que se ejecuta automáticamente al instanciar el nodo.

### Inicializa el nodo de ROS 2 con su respectivo nombre
**super().__init__("scara_tray_line_node")** #<----- Llama al constructor de Node asignando el nombre "scara_tray_line_node".

### Configuración del tópico objetivo para el controlador de trayectoria
**topic_name = "/scara_trajectory_controller/joint_trajectory"** #<----- Variable que almacena el tópico de ROS 2 al que se enviará la trayectoria.

**self.joints_ = ['link_1_joint', 'link_2_joint', 'link_3_joint']** #<----- Arreglo que define los nombres de las tres articulaciones del robot SCARA.

### Inicialización de variables de control y tiempo de ejecución
**self.lamda_ = 0** #<----- Inicializa un parámetro de control (comúnmente usado para interpolación lineal $\lambda$).

**self.Tiempo_ejec_ = 10** #<----- Define la duración estimada para la ejecución de la trayectoria (10 segundos).

**self.scara_tray_pub_ = self.create_publisher(JointTrajectory, topic_name, 10)** #<----- Crea el publicador de ROS 2 para enviar mensajes JointTrajectory con QoS de 10.

**self.tray_timer_ = self.create_timer(1, self.trayectory_cbck)** #<----- Configura un temporizador para ejecutar la función trayectory_cbck cada 1 segundo.

**self.get_logger().info('Scara activo, trayectoria linea recta')** #<----- Imprime un mensaje informativo en la terminal de ROS 2 indicando éxito.
 


### Función callback del temporizador para calcular y publicar la trayectoria
**def trayectory_cbck(self):** #<----- Define la función callback que el temporizador ejecuta periódicamente.

**trayectory_msg = JointTrajectory()** #<----- Instancia un objeto vacío del tipo de mensaje JointTrajectory.

**trayectory_msg.joint_names = self.joints_** #<----- Asigna la lista con los nombres de las articulaciones al mensaje.

**point = JointTrajectoryPoint()** #<----- Crea un objeto vacío para almacenar la posición, velocidad y tiempo del punto actual.

### Verifica si el parámetro actual está dentro del tiempo límite de ejecución
**if self.lamda_ <= self.Tiempo_ejec_:** #<----- Condicional que valida si el tiempo actual ($\lambda$) es menor o igual al tiempo total programado.

### Configuración del punto inicial de la trayectoria recta (Posición y Orientación)
   *x_1 = 0.1* #<----- Define la coordenada X del punto inicial de la trayectoria en el espacio cartesiano.

   *y_1 = 0.6* #<----- Define la coordenada Y del punto inicial de la trayectoria.

   *theta_1 = 0* #<----- Define la orientación angular inicial del efector final del robot.

### Configuración del punto final de la trayectoria recta (Posición y Orientación)
   *x_2 = 0.3* #<----- Define la coordenada X del punto final de la trayectoria recta.

   *y_2 = -0.6* #<----- Define la coordenada Y del punto final de la trayectoria recta.

   *theta_2 = 1.57* #<----- Define la orientación angular final del efector final (en radianes, equivalente a ~90°).


### Calcula la cinemática inversa para el instante actual usando una función externa
**solucion = invk_sol(self.lamda_, x_1, y_1, theta_1, x_2, y_2, theta_2)** #<----- Llama a una función externa de cinemática inversa pasando los puntos, orientaciones y el paso actual.

**point.positions = solucion** #<----- Asigna los ángulos calculados (articulaciones) a la propiedad de posiciones del punto.

**point.time_from_start = Duration(sec=1)** #<----- Define que este punto de la trayectoria debe alcanzarse en un tiempo de 1 segundo.

**trayectory_msg.points.append(point)** #<----- Añade el punto configurado a la lista de puntos del mensaje de trayectoria.

**self.scara_tray_pub_.publish(trayectory_msg)** #<----- Publica el mensaje completo en el tópico de ROS 2 para que el robot ejecute el movimiento.

**self.get_logger().info("Postura actual {}".format(solucion))** #<----- Imprime en la consola de ROS 2 los ángulos calculados para la postura actual del robot.

**time.sleep(2)** #<----- Pausa la ejecución del hilo actual por 2 segundos para dar tiempo físico al movimiento.

**self.lamda_ += 1** #<----- Incrementa en 1 unidad el parámetro de tiempo/interpolación para avanzar al siguiente punto.

### Condición en caso de que el tiempo de ejecución supere el límite de 10 segundos
**elif self.lamda_ > 10:** #<----- Evalúa si el parámetro de interpolación o tiempo ($\lambda$) ha superado el límite de 10 segundos.

### Configura las posiciones angulares de reposo (cero) para cada articulación
  *link_1_joint = 0* #<----- Asigna un ángulo de 0 radianes a la primera articulación del robot.

  *link_2_joint = 0* #<----- Asigna un ángulo de 0 radianes a la segunda articulación del robot.

  *link_3_joint = 0* #<----- Asigna un ángulo de 0 radianes a la tercera articulación (o eje lineal) del robot.

**return [float(link_1_joint), float(link_2_joint), float(link_3_joint)]** #<----- Convierte los valores numéricos a tipo flotante y los retorna dentro de una lista compatible con el mensaje.


### Función para calcular la cinemática inversa interpolada en línea recta
**def invk_sol(param, x_in, y_in, theta_in, x_fin, y_fin, theta_fin):** #<----- Define la función de cinemática inversa con los parámetros cartesianos de entrada y salida.

### Tiempo total asignado para la ejecución de la trayectoria
**Tiempo_ejec_ = 10** #<----- Define localmente el tiempo máximo de ejecución para la normalización (10 segundos).

### Dimensiones físicas de los eslabones del robot SCARA
**L_1 = 0.5** #<----- Longitud física del primer eslabón del robot SCARA (en metros).

**L_2 = 0.5** #<----- Longitud física del segundo eslabón del robot.

**L_3 = 0.3** #<----- Longitud física del tercer eslabón o herramienta.

### Interpolación lineal paramétrica para obtener la trayectoria recta en el espacio cartesiano
**x_P = x_in + (param/Tiempo_ejec_)*(x_fin - x_in)** #<----- Interpolación lineal en X: Calcula la posición intermedia en el eje X para el instante actual.

**y_P = y_in + (param/Tiempo_ejec_)*(y_fin - y_in)** #<----- Interpolación lineal en Y: Calcula la posición intermedia en el eje Y para el instante actual.

**theta_P = theta_in + (param/Tiempo_ejec_)*(theta_fin - theta_in)** #<----- Interpolación de orientación: Calcula el ángulo del efector final para el instante actual.

### Desacoplamiento cinemático para aislar la posición de la estructura principal (articulación 3)
**x_3 = x_P - L_3*cos(theta_P)** #<----- Desacoplamiento geométrico: Calcula la coordenada X de la muñeca / articulación 3.

**y_3 = y_P - L_3*sin(theta_P)** #<----- Desacoplamiento geométrico: Calcula la coordenada Y de la muñeca / articulación 3.

### Cálculo del ángulo de la articulación 2 empleando la ley de cosenos
**theta_2 = acos((pow(x_3, 2)+pow(y_3, 2)-pow(L_1, 2)-pow(L_2, 2))/(2*L_1*L_2))** #<----- Ley de Cosenos: Calcula el ángulo de la segunda articulación ($\theta_2$).


### Cálculo geométrico de ángulos auxiliares para obtener la articulación 1
**beta = atan2(y_3, x_3)** #<----- Calcula el ángulo polar de la posición de la muñeca respecto al origen.

**psi = acos((pow(x_3, 2)+pow(y_3, 2)+pow(L_1, 2)-pow(L_2, 2))/(2*L_1*sqrt(pow(x_3, 2)+pow(y_3, 2))))** #<----- Geometría del triángulo: Calcula el ángulo interno del triángulo formado por los eslabones.

### Determinación del ángulo de la primera articulación
**theta_1 = beta - psi** #<----- Ángulo de articulación 1: Resta geométrica para hallar la orientación del primer eslabón.

### Determinación del ángulo de la tercera articulación para corregir la orientación final
**theta_3 = theta_P - theta_1 - theta_2** #<----- Ángulo de articulación 3: Deducción matemática para orientar la herramienta final.

### Retorna el conjunto de soluciones articulares requeridas por el controlador del robot
**return [float(theta_1), float(theta_2), float(theta_3)]** #<----- Devuelve las tres variables articulares calculadas en formato de lista con tipos flotantes.

### Función principal encargada de gestionar el ciclo de vida del nodo
**def main(args=None):** #<----- Define la función principal main que coordina el ciclo de vida del nodo.

### Inicializar el sistema de comunicaciones de ROS 2
**rclpy.init(args=args)** #<----- Inicializa el sistema de comunicación y la capa de abstracción de ROS 2.


### Crear una instancia del nodo del robot SCARA, activando publicadores y temporizadores 

   **node = ScaraTrayLineNode()** #<----- Crea una instancia de nuestra clase, activando el constructor y sus configuraciones.

### Bloquear el script y mantiene el nodo activo procesando eventos continuamente

   **rclpy.spin(node)** #<----- Mantiene el nodo vivo y en ejecución, procesando los callbacks del temporizador continuamente

### Finalizar las comunicaciones y libera limpiamente los recursos de ROS 2 al cerrar el nodo
 
   **rclpy.shutdown()** #<----- Realiza un cierre limpio, liberando los recursos y destruyendo el nodo al detener el programa.

### Punto de entrada estándar para ejecutar el script desde la terminal

if __name__ == "__main__":
    main()

## Funciones Principales del Código

1. **Inicialización y Registro del Nivel en ROS 2:** El programa se registra dentro del ecosistema de ROS 2 bajo el nombre de `scara_tray_line_node`. Configura un canal de comunicación (Publisher) apuntando al tópico `/scara_trajectory_controller/joint_trajectory`, el cual interactúa directamente con los controladores de los motores (normalmente `ros2_control`).

2. **Generador de Tiempo e Interpolación Lineal:** Utiliza un temporizador (Timer) integrado que se ejecuta automáticamente cada segundo. El código maneja una variable paramétrica de tiempo ($\lambda$) que va desde $0$ hasta $10$ segundos. Con este parámetro, el script realiza una interpolación lineal cartesiana para fraccionar una línea recta que va desde un punto inicial ($P_1(x_1, y_1, \theta_1)$) hasta un punto final ($P_2(x_2, y_2, \theta_2)$).

3. **Cálculo en Tiempo Real de Cinemática Inversa:** Para cada segundo que pasa, el script toma la posición cartesiana intermedia calculada y ejecuta la función `invk_sol`. Esta función aplica ecuaciones trigonométricas y geométricas (como la Ley de Cosenos y la función Arcotangente de dos parámetros `atan2`) basadas en las dimensiones reales del robot ($L_1, L_2, L_3$) para traducir las coordenadas ($(X, Y)$) del espacio físico a los ángulos específicos en radianes que requiere cada motor ($\theta_1, \theta_2, \theta_3$).

4. **Empaquetado y Publicación de Mensajes Estándar:** Una vez que obtiene los ángulos de los motores, los empaqueta dentro de un objeto estructurado de tipo `JointTrajectoryPoint`. Este punto se añade al cuerpo de un mensaje `JointTrajectory`, se le asigna un tiempo de llegada de 1 segundo, y se envía a los motores a través del publicador.

5. **Rutina de Seguridad y Reposo (Home):** Si el contador de tiempo supera el límite establecido de la trayectoria ($10$ segundos), el código entra en una condición de seguridad (`elif self.lamda_ > 10`), donde redefine los objetivos de todas las articulaciones a $0$ radianes. Esto actúa como un comando de retorno a la posición de reposo o configuración "Home" del manipulador.

6. **Gestión del Ciclo de Vida del Programa:** A través de la función `main`, el script controla el encendido del entorno ROS 2, mantiene el programa en un bucle infinito de escucha activa (`rclpy.spin`) para que el temporizador no se detenga, y asegura un apagado controlado de los recursos de hardware (`rclpy.shutdown`) en caso de que el usuario interrumpa la ejecución.


# Documentación del Nodo: `doftbot_sequence_py.py`

Este script inicializa un nodo de **ROS 2** en Python diseñado para controlar las trayectorias de un robot tipo SCARA en línea recta.

## Descripción del Bloque

## Resumen de las Funciones Principales del Código (`DofbotControlNode`)

Este script implementa un **nodo de control secuencial automatizado** en ROS 2 para un brazo manipulador **Dofbot de 5 grados de libertad** equipado con una **pinza (gripper) simétrica de 6 articulaciones**. Su objetivo principal es guiar al robot a través de una rutina cíclica preprogramada de tareas de sujeción, traslado y liberación de objetos en un espacio tridimensional.

**#!/usr/bin/env python3** #<----- **Shebang**: Indica al sistema operativo que ejecute el archivo con Python 3.


### Importación de librerías de ROS 2 y tipos de mensajes

**import rclpy**   #<-----  Importa la librería cliente principal de ROS 2 para Python. 

**from rclpy.node import Node**  #<----- Importa la clase base `Node` para crear y gestionar nodos en ROS 2.

### Interfaces de mensajes de ROS 2 para trayectorias de articulaciones y duraciones

**from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint**  #<----- Importa los tipos de mensajes necesarios para enviar comandos de trayectorias multieje a las articulaciones.

**from builtin_interfaces.msg import Duration**  #<-----Importa el tipo de mensaje para definir duraciones de tiempo específicas de ROS 2.

### Importación de librerías matemáticas y de tiempo

**import time** #<-----Importa el módulo estándar de Python para gestionar funciones de tiempo.

**from math import cos, sin, acos, asin, atan2, sqrt** #<-----Importa funciones trigonométricas y matemáticas para la cinemática del robot.

###  Declaración de la clase del nodo para el control del robot Dofbot

**class DofbotControlNode(Node):**   #<----- Define la clase del nodo que hereda de la clase base Node de ROS 2.

**def __init__(self):**  #<----- Método constructor que se ejecuta automáticamente al instanciar el nodo.
        
### Inicializa el nodo de ROS 2 con su identificador oficial
 
**super().__init__("dofbot_tray_control_node")**  #<----- Inicializa la clase padre asignando al nodo el nombre "dofbot_tray_control_node".

### Parámetro de interpolación lineal (\(\lambda\)) inicializado en cero
 
self.lamda_ = 0  #<----- Inicializa el parámetro de interpolación o control lineal \(\lambda \) en 0.

### Tópicos asignados a los controladores de trayectoria del brazo y del gripper

**topic_dofbot_ = "/dofbot_trajectory_controller/joint_trajectory"**  #<----- Guarda la ruta del tópico para el controlador de trayectoria del brazo de 5 ejes.

**topic_gripper_ = "/dofbot_gripper_controller/joint_trajectory"**  #<----- Guarda la ruta del tópico para el controlador de trayectoria de la pinza (gripper).

### Publicador y configuración de las 5 articulaciones principales del brazo robótico

**self.dofbot_publisher_ = self.create_publisher(JointTrajectory, topic_dofbot_, 10)**  #<----- Crea el publicador de ROS 2 para el brazo usando mensajes JointTrajectory (QoS=10). Especifica el tipo de mensaje, tópico y el tamaño de cola.

**self.dofbot_joints_ = ['arm_joint_01', 'arm_joint_02','arm_joint_03', 'arm_joint_04', 'arm_joint_05']** #<----- Crea un arreglo de cadenas con los nombres de las 5 articulaciones secuenciales del brazo.

### Publicador y configuración de los 6 eslabones/articulaciones del gripper
        
**self.gripper_publisher_ = self.create_publisher(JointTrajectory, topic_gripper_, 10)**  #<----- Crea el publicador de ROS 2 exclusivo para controlar el actuador final o pinza. Define los parámetros de configuración para el publicador de la pinza.
        
**self.gripper_joints_ = ['grip_joint', 'rfinger_joint_01','rfinger_joint_02', 'lfinger_grip_joint_01','lfinger_grip_joint_02', 'lfinger_grip_joint_03']**  #<----- Arreglo que enumera los nombres de las 6 articulaciones mecánicas que componen el gripper.
        
### Activar el temporizador del nodo e imprimir un mensaje de confirmación en la consola.         
**self.timer_ = self.create_timer(0.5, self.timer_callback)**  #<----- Crea un temporizador en ROS 2 que ejecuta la función self.timer_callback automáticamente cada 0.5 segundos (frecuencia de 2 Hz).
        
**self.get_logger().info('Nodo de control del dofbot en funcionamiento')**  #<----- Envía una alerta informativa a la terminal para confirmar visualmente que el script está corriendo de manera correcta.
        

###  Función callback para gestionar periódicamente las trayectorias de los actuadores

**def timer_callback(self):**  #<----- Declara la función de retorno (callback) que se ejecuta automáticamente cada vez que el temporizador cumple su ciclo.

###  --- Configuración del mensaje para el brazo robótico (Dofbot) ---
   ### Crea la estructura vacía del mensaje de trayectoria para las articulaciones del brazo
    
**dofbot_msg = JointTrajectory()**  #<----- instancia un objeto vacío del tipo JointTrajectory destinado al control del brazo.
        
**dofbot_msg.joint_names = self.dofbot_joints_**  #<----- Asigna la lista de nombres de las 5 articulaciones del brazo al mensaje.
        
**dofbot_point = JointTrajectoryPoint()**  #<----- Crea un objeto vacío para guardar los parámetros de posición y tiempo del brazo en el instante actual.

 ###  --- Configuración del mensaje para la pinza (Gripper) ---
   ###  Crea la estructura vacía del mensaje de trayectoria para el gripper

**gripper_msg = JointTrajectory()**  #<-----  Instancia un objeto vacío del tipo JointTrajectory destinado al control de la pinza.
        
**gripper_msg.joint_names = self.gripper_joints_**  #<----- Asigna la lista de nombres de las 6 articulaciones de la pinza al mensaje.
        
**gripper_point = JointTrajectoryPoint()**  #<----- Crea un objeto vacío para guardar los parámetros de posición y tiempo de la pinza en el instante actual.




 ### Estado inicial del ciclo de control (\(\lambda = 0\)). Apertura Completa de la Pinza

**if self.lamda_ == 0:** #<----- Condicional que valida si la secuencia está en su paso o estado inicial ($\lambda = 0$).
   
 ### Acción designada: Abrir el gripper    
    
**gstate = 1.57** #<----- Define el valor objetivo en radianes para la apertura (equivalente a unos 90° o $\pi/2$).
    
**gripper_st = gripper_state(gstate)** #<----- Llama a una función externa para mapear ese valor único a los 6 eslabones cinemáticos de la pinza.
    
**gripper_point.positions = gripper_st** #<----- Asigna el arreglo de posiciones angulares calculado al punto de trayectoria del gripper.
    
**gripper_point.time_from_start = Duration(sec=1)** #<----- Define que el gripper debe completar la apertura en un tiempo de 1 segundo.
    
**gripper_msg.points.append(gripper_point)** #<----- Inserta este punto configurado dentro de la lista de puntos del mensaje de trayectoria.
    
**self.gripper_publisher_.publish(gripper_msg)** #<----- Envía el mensaje completo al controlador ROS 2 de la pinza para realizar el movimiento físico.

#### Registros informativos del estado y configuración angular en la consola de ROS 2
    
 **self.get_logger().info('Gripper open')** #<----- Envía una alerta informativa a la terminal indicando que la pinza se ha abierto.
    
**self.get_logger().info('poture {}'.format(gripper_st))** #<----- Imprime en la consola el arreglo de ángulos actuales enviados al gripper.
    
**time.sleep(5)** #<----- Detiene por completo el hilo durante 5 segundos para asegurar que la pinza se abra físicamente del todo.
    
**self.lamda_ += 1** #<----- Incrementa en 1 unidad el parámetro de control ($\lambda$) para avanzar al siguiente estado de la rutina


### --- Lógica del Segundo Estado ($\lambda = 1$) ---

### Verificación del paso de sujeción de la secuencia temporal
**elif self.lamda_ == 1:** #<----- Condicional que valida si la secuencia se encuentra en su segundo paso o estado ($\lambda = 1$).

### Acción designada: Cerra el gripper
### Configuración del ángulo objetivo para el cierre
**gstate_2 = 0** #<----- Define el valor objetivo en radianes para cerrar la pinza (posición angular cero).
    
### Mapeo cinemático para el cierre total del actuador
**gripper_st = gripper_state(gstate_2)** #<----- Llama a la función externa para mapear la posición de cierre a las 6 articulaciones mecánicas de la pinza.
    
### Carga de datos articulares y restricciones de tiempo en el punto
**gripper_point.positions = gripper_st** #<----- Asigna el arreglo de posiciones angulares de cierre al punto de trayectoria del gripper.
    
**gripper_point.time_from_start = Duration(sec=1)** #<----- Define que el gripper debe completar la acción de cierre en un lapso de 1 segundo.
    
### Empaquetado y transferencia de comandos al hardware
**gripper_msg.points.append(gripper_point)** #<----- Añade este punto de cierre configurado a la lista de puntos del mensaje de trayectoria.
    
**self.gripper_publisher_.publish(gripper_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 de la pinza ejecute el movimiento.
    
### Telemetría del estado de sujeción en la consola
**self.get_logger().info('Gripper close')** #<----- Envía un mensaje informativo a la terminal indicando que la pinza se ha cerrado.
    
**self.get_logger().info('poture {}'.format(gripper_st))** #<----- Imprime en la consola el arreglo de ángulos actuales enviados al gripper para confirmar el cierre.
    
### Sincronización física del actuador y actualización del contador
**time.sleep(5)** #<----- Realiza una pausa obligatoria en el hilo por 5 segundos para que la pinza complete físicamente su recorrido.
    
**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para dar paso al siguiente estado de la rutina.


### --- Lógica del Tercer Estado ($\lambda = 2$) ---

### Verificación del tercer paso de la secuencia temporal
**elif self.lamda_ == 2:** #<----- Condicional que valida si la secuencia se encuentra en su tercer paso o estado ($\lambda = 2$).

### Open gripper
### Configuración del ángulo objetivo para la reapertura de la pinza
**gstate = 1.57** #<----- Define el valor objetivo en radianes para la apertura (equivalente a unos 90° o $\pi/2$).

### Mapeo cinemático para la apertura completa del actuador final
**gripper_st = gripper_state(gstate)** #<----- Llama a la función externa para mapear la posición de apertura a las 6 articulaciones mecánicas de la pinza.

### Carga de datos articulares y restricciones de tiempo en el punto
**gripper_point.positions = gripper_st** #<----- Asigna el arreglo de posiciones angulares de apertura al punto de trayectoria del gripper.

**gripper_point.time_from_start = Duration(sec=1)** #<----- Define que el gripper debe completar la acción de apertura en un lapso de 1 segundo.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**gripper_msg.points.append(gripper_point)** #<----- Añade este punto de apertura configurado a la lista de puntos del mensaje de trayectoria.

**self.gripper_publisher_.publish(gripper_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 de la pinza ejecute el movimiento.

### Telemetría y registro del estado de liberación en la consola
**self.get_logger().info('Gripper open')** #<----- Envía un mensaje informativo a la terminal indicando que la pinza se ha abierto nuevamente.

**self.get_logger().info('poture {}'.format(gripper_st))** #<----- Imprime en la consola el arreglo de ángulos actuales enviados al gripper para confirmar la acción.

### Sincronización física del actuador y actualización del contador secuencial
**time.sleep(5)** #<----- Realiza una pausa obligatoria en el hilo por 5 segundos para que la pinza complete físicamente su recorrido.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para dar paso al siguiente estado de la rutina.


### --- Lógica del Cuarto Estado ($\lambda = 3$) ---

### Verificación del cuarto paso de la secuencia temporal
**elif self.lamda_ == 3:** #<----- Condicional que valida si la secuencia se encuentra en su cuarto paso o estado ($\lambda = 3$).

### Primera postura
### Definición de las coordenadas de posición cartesiana objetivo ($x_1, y_1, z_1$)
**x_1 = 0.2** #<----- Coordenada objetivo en el eje X para la primera postura del brazo.

**y_1 = 0.0** #<----- Coordenada objetivo en el eje Y para la primera postura del brazo.

**z_1 = 0.05** #<----- Coordenada objetivo en el eje Z (altura) para la primera postura del brazo.

### Definición de los ángulos de orientación para el efector final y la herramienta
**theta_p_1 = 3.1416*(3/4)** #<----- Ángulo de paso (pitch) deseado en radianes para la muñeca del robot.

**theta_g_1 = 0.0** #<----- Ángulo de giro (roll) o rotación final de la herramienta en radianes.

### Mapeo cinemático inverso para el brazo robótico Dofbot
**solution_pos = dofbot_ink(x_1, y_1, z_1, theta_p_1, theta_g_1)** #<----- Llama a la función de cinemática inversa para traducir las coordenadas cartesianas a los ángulos de los 5 motores.

### Carga de datos articulares y restricciones de tiempo en el punto
**dofbot_point.positions = solution_pos** #<----- Asigna el arreglo de posiciones angulares calculado al punto de trayectoria del brazo.

**dofbot_point.time_from_start = Duration(sec=2)** #<----- Define que el brazo debe completar el movimiento hacia esta postura en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**dofbot_msg.points.append(dofbot_point)** #<----- Añade este punto de trayectoria configurado a la lista de puntos del mensaje del brazo.

**self.dofbot_publisher_.publish(dofbot_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 mueva los ejes del brazo.

### Telemetría y registro de la postura articular en la consola
**self.get_logger().info('poture {}'.format(solution_pos))** #<----- Imprime en la terminal de ROS 2 el arreglo de ángulos calculados para su verificación.

### Sincronización física del manipulador y actualización del contador secuencial
**time.sleep(15)** #<----- Realiza una pausa larga de 15 segundos en el hilo para dar tiempo suficiente a que el brazo complete su trayectoria física de forma segura.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para avanzar al siguiente estado de la rutina.


### --- Lógica del Quinto Estado ($\lambda = 4$) ---

### Verificación del quinto paso de la secuencia temporal
**elif self.lamda_ == 4:** #<----- Condicional que valida si la secuencia se encuentra en su quinto paso o estado ($\lambda = 4$).

### Close gripper
### Configuración del ángulo objetivo para el cierre de la pinza
**gstate_2 = 0** #<----- Define el valor objetivo en radianes para cerrar la pinza (posición angular cero).

### Mapeo cinemático para el cierre total del actuador final
**gripper_st = gripper_state(gstate_2)** #<----- Llama a la función externa para mapear la posición de cierre a las 6 articulaciones mecánicas de la pinza.

### Carga de datos articulares y restricciones de tiempo en el punto
**gripper_point.positions = gripper_st** #<----- Asigna el arreglo de posiciones angulares de cierre al punto de trayectoria del gripper.

**gripper_point.time_from_start = Duration(sec=2)** #<----- Define que el gripper debe completar la acción de cierre en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**gripper_msg.points.append(gripper_point)** #<----- Añade este punto de cierre configurado a la lista de puntos del mensaje de trayectoria.

**self.gripper_publisher_.publish(gripper_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 de la pinza ejecute el movimiento.

### Telemetría y registro del estado de sujeción en la consola
**self.get_logger().info('Gripper close')** #<----- Envía un mensaje informativo a la terminal indicando que la pinza se ha cerrado.

**self.get_logger().info('poture {}'.format(gripper_st))** #<----- Imprime en la consola el arreglo de ángulos actuales enviados al gripper para confirmar el cierre.

### Sincronización física del actuador y actualización del contador secuencial
**time.sleep(10)** #<----- Realiza una pausa obligatoria en el hilo por 10 segundos para dar tiempo a que la pinza termine de sujetar físicamente el objeto de forma segura.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para dar paso al siguiente estado de la rutina.


### --- Lógica del Sexto Estado ($\lambda = 5$) ---

### Verificación del sexto paso de la secuencia temporal
**elif self.lamda_ == 5:** #<----- Condicional que valida si la secuencia se encuentra en su sexto paso o estado ($\lambda = 5$).

### Segunda postura
### Definición de las coordenadas de posición cartesiana objetivo ($x_2, y_2, z_2$)
**x_2 = 0.15** #<----- Coordenada objetivo en el eje X para la segunda postura del brazo.

**y_2 = 0.0** #<----- Coordenada objetivo en el eje Y para la segunda postura del brazo.

**z_2 = 0.11** #<----- Coordenada objetivo en el eje Z (altura) para la segunda postura del brazo.

### Definición de los ángulos de orientación para el efector final y la herramienta
**theta_p_2 = 3.1416*(3/4)** #<----- Ángulo de paso (pitch) deseado en radianes para la muñeca del robot en este estado.

**theta_g_2 = 0** #<----- Ángulo de giro (roll) o rotación final de la herramienta en radianes.

### Mapeo cinemático inverso para el brazo robótico Dofbot
**solution_pos = dofbot_ink(x_2, y_2, z_2, theta_p_2, theta_g_2)** #<----- Llama a la función de cinemática inversa para traducir las nuevas coordenadas cartesianas a los ángulos de los 5 motores.

### Carga de datos articulares y restricciones de tiempo en el punto
**dofbot_point.positions = solution_pos** #<----- Asigna el arreglo de posiciones angulares calculado al punto de trayectoria del brazo.

**dofbot_point.time_from_start = Duration(sec=2)** #<----- Define que el brazo debe completar el movimiento hacia esta segunda postura en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**dofbot_msg.points.append(dofbot_point)** #<----- Añade este punto de trayectoria configurado a la lista de puntos del mensaje del brazo.

**self.dofbot_publisher_.publish(dofbot_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 mueva los ejes del brazo a la nueva postura.

### Telemetría y registro de la postura articular en la consola
**self.get_logger().info('poture {}'.format(solution_pos))** #<----- Imprime en la terminal de ROS 2 el arreglo de ángulos calculados para su verificación.

### Sincronización física del manipulador y actualización del contador secuencial
**time.sleep(15)** #<----- Realiza una pausa larga de 15 segundos en el hilo para dar tiempo suficiente a que el brazo complete su trayectoria física de forma segura.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para avanzar al siguiente estado de la rutina.


### --- Lógica del Séptimo Estado ($\lambda = 6$) ---

### Verificación del séptimo paso de la secuencia temporal
**elif self.lamda_ == 6:** #<----- Condicional que valida si la secuencia se encuentra en su séptimo paso o estado ($\lambda = 6$).

### Tercer postura
### Definición de las coordenadas de posición cartesiana objetivo ($x_3, y_3, z_3$)
**x_3 = 0.15** #<----- Coordenada objetivo en el eje X para la tercera postura del brazo.

**y_3 = 0.15** #<----- Coordenada objetivo en el eje Y para la tercera postura del brazo.

**z_3 = 0.11** #<----- Coordenada objetivo en el eje Z (altura) para la tercera postura del brazo.

### Definición de los ángulos de orientación para el efector final y la herramienta
**theta_p_3 = 3.1416*(3/4)** #<----- Ángulo de paso (pitch) deseado en radianes para la muñeca del robot en este estado.

**theta_g_3 = 0** #<----- Ángulo de giro (roll) o rotación final de la herramienta en radianes.

### Mapeo cinemático inverso para el brazo robótico Dofbot
**solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)** #<----- Llama a la función de cinemática inversa para traducir las nuevas coordenadas cartesianas a los ángulos de los 5 motores.

### Carga de datos articulares y restricciones de tiempo en el punto
**dofbot_point.positions = solution_pos** #<----- Asigna el arreglo de posiciones angulares calculado al punto de trayectoria del brazo.

**dofbot_point.time_from_start = Duration(sec=2)** #<----- Define que el brazo debe completar el movimiento hacia esta tercera postura en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**dofbot_msg.points.append(dofbot_point)** #<----- Añade este punto de trayectoria configurado a la lista de puntos del mensaje del brazo.

**self.dofbot_publisher_.publish(dofbot_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 mueva los ejes del brazo a la tercera postura.

### Telemetría y registro de la postura articular en la consola
**self.get_logger().info('poture {}'.format(solution_pos))** #<----- Imprime en la terminal de ROS 2 el arreglo de ángulos calculados para su verificación.

### Sincronización física del manipulador y actualización del contador secuencial
**time.sleep(15)** #<----- Realiza una pausa larga de 15 segundos en el hilo para dar tiempo suficiente a que el brazo complete su trayectoria física de forma segura.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para avanzar al siguiente estado de la rutina.


### --- Lógica del Octavo Estado ($\lambda = 7$) ---

### Verificación del octavo paso de la secuencia temporal
**elif self.lamda_ == 7:** #<----- Condicional que valida si la secuencia se encuentra en su octavo paso o estado ($\lambda = 7$).

#### Cuarta postura
### Definición de las coordenadas de posición cartesiana objetivo ($x_3, y_3, z_3$)
**x_3 = 0.15** #<----- Coordenada objetivo en el eje X para la cuarta postura del brazo.

**y_3 = 0.15** #<----- Coordenada objetivo en el eje Y para la cuarta postura del brazo.

**z_3 = 0.05** #<----- Coordenada objetivo en el eje Z (altura) para la cuarta postura del brazo.

### Definición de los ángulos de orientación para el efector final y la herramienta
**theta_p_3 = 3.1416*(3/4)** #<----- Ángulo de paso (pitch) deseado en radianes para la muñeca del robot en este estado.

**theta_g_3 = 0** #<----- Ángulo de giro (roll) o rotación final de la herramienta en radianes.

### Mapeo cinemático inverso para el brazo robótico Dofbot
**solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)** #<----- Llama a la función de cinemática inversa para traducir las nuevas coordenadas cartesianas a los ángulos de los 5 motores.

### Carga de datos articulares y restricciones de tiempo en el punto
**dofbot_point.positions = solution_pos** #<----- Asigna el arreglo de posiciones angulares calculado al punto de trayectoria del brazo.

**dofbot_point.time_from_start = Duration(sec=2)** #<----- Define que el brazo debe completar el movimiento hacia esta cuarta postura en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**dofbot_msg.points.append(dofbot_point)** #<----- Añade este punto de trayectoria configurado a la lista de puntos del mensaje del brazo.

**self.dofbot_publisher_.publish(dofbot_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 mueva los ejes del brazo a la cuarta postura.

### Telemetría y registro de la postura articular en la consola
**self.get_logger().info('poture {}'.format(solution_pos))** #<----- Imprime en la terminal de ROS 2 el arreglo de ángulos calculados para su verificación.

### Sincronización física del manipulador y actualización del contador secuencial
**time.sleep(15)** #<----- Realiza una pausa larga de 15 segundos en el hilo para dar tiempo suficiente a que el brazo complete su trayectoria física de forma segura.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para avanzar al siguiente estado de la rutina.


### --- Lógica del Noveno Estado ($\lambda = 8$) ---

### Verificación del noveno paso de la secuencia temporal
**elif self.lamda_ == 8:** #<----- Condicional que valida si la secuencia se encuentra en su noveno paso o estado ($\lambda = 8$).

# Open gripper
### Configuración del ángulo objetivo para la reapertura de la pinza
**gstate = 1.57** #<----- Define el valor objetivo en radianes para la apertura (equivalente a unos 90° o $\pi/2$).

### Mapeo cinemático para la apertura completa del actuador final
**gripper_st = gripper_state(gstate)** #<----- Llama a la función externa para mapear la posición de apertura a las 6 articulaciones mecánicas de la pinza.

### Carga de datos articulares y restricciones de tiempo en el punto
**gripper_point.positions = gripper_st** #<----- Asigna el arreglo de posiciones angulares de apertura al punto de trayectoria del gripper.

**gripper_point.time_from_start = Duration(sec=2)** #<----- Define que el gripper debe completar la acción de apertura en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**gripper_msg.points.append(gripper_point)** #<----- Añade este punto de apertura configurado a la lista de puntos del mensaje de trayectoria.

**self.gripper_publisher_.publish(gripper_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 de la pinza ejecute el movimiento.

### Telemetría y registro del estado de liberación en la consola
**self.get_logger().info('Gripper open')** #<----- Envía un mensaje informativo a la terminal indicando que la pinza se ha abierto de nuevo.

**self.get_logger().info('poture {}'.format(gripper_st))** #<----- Imprime en la consola el arreglo de ángulos actuales enviados al gripper para confirmar la acción.

### Sincronización física del actuador y actualización del contador secuencial
**time.sleep(10)** #<----- Realiza una pausa de 10 segundos en el hilo para asegurar que la pinza suelte físicamente la carga por completo.

**self.lamda_ += 1** #<----- Incrementa el parámetro de control ($\lambda$) en una unidad para avanzar al siguiente estado de la rutina.


### --- Lógica del Décimo Estado ($\lambda = 9$) ---

### Verificación del décimo paso de la secuencia temporal (Posición de Reposo / Home)
**elif self.lamda_ == 9:** #<----- Condicional que valida si la secuencia se encuentra en su décimo paso o estado ($\lambda = 9$).

### Definición de la postura angular de reposo para todos los motores del brazo
**solution_pos = [ float(0.0), float(0.0), float(0.0), float(0.0), float(0.0)]** #<----- Define una lista con valores en cero convertidos a flotantes para regresar el brazo a su posición "Home".

### Carga de datos articulares y restricciones de tiempo en el punto
**dofbot_point.positions = solution_pos** #<----- Asigna el arreglo de posiciones angulares de reposo al punto de trayectoria del brazo.

**dofbot_point.time_from_start = Duration(sec=2)** #<----- Define que el brazo debe completar el retorno a la posición de reposo en un lapso de 2 segundos.

### Empaquetado y transferencia de comandos al hardware mediante tópicos
**dofbot_msg.points.append(dofbot_point)** #<----- Añade este punto de trayectoria a la lista de puntos del mensaje del brazo.

**self.dofbot_publisher_.publish(dofbot_msg)** #<----- Publica el mensaje en el tópico para que el controlador ROS 2 mueva los ejes a la posición inicial segura.

### Telemetría y registro de la postura articular en la consola
**self.get_logger().info('poture {}'.format(solution_pos))** #<----- Imprime en la terminal de ROS 2 el arreglo de ángulos en cero para confirmar el retorno a Home.

### Sincronización física final del manipulador antes de culminar la rutina
**time.sleep(10)** #<----- Realiza una pausa obligatoria de 10 segundos en el hilo para asegurar que todo el brazo complete físicamente su trayectoria de regreso.


### --- Función Externa para la Cinemática Inversa (`dofbot_ink`) ---

### Declaración de la función y definición de parámetros cartesianos de entrada
**def dofbot_ink(x_P, y_P, z_P, theta_1_P, theta_g):** #<----- Define la función que calculará los ángulos de las articulaciones del brazo Dofbot en base a las posiciones y orientaciones cartesianas deseadas.

# Parametros
### Especificación de las dimensiones físicas reales de los eslabones del robot
*z_0_1 = 0.105* #<----- Define la altura física del eslabón de la base o primer eje del robot (en metros).

*L_1 = 0.084* #<----- Define la longitud del primer eslabón móvil del brazo (en metros).

*L_2 = 0.084* #<----- Define la longitud del segundo eslabón móvil del brazo (en metros).

*L_3 = 0.115* #<----- Define la longitud del tercer eslabón o sección que sostiene la herramienta final (en metros).


### Cálculo del ángulo de la base mediante arcotangente
*theta_1 = atan2(y_P, x_P)* #<----- Calcula el ángulo $\theta_1$ para orientar la base del robot hacia las coordenadas ($X, Y$).

### Desacoplamiento cinemático y proyección en los ejes planar y vertical
*aux_x = sqrt(pow(x_P, 2) + pow(y_P, 2)) - L_3*sin(theta_1_P)** #<----- Proyección intermedia sobre el plano horizontal restando el efecto de la muñeca.

aux_z = z_P - z_0_1 -L_3*cos(theta_1_P) #<----- Proyección intermedia sobre el eje vertical $Z$ descontando la altura de la base y la muñeca.

### Determinación de la norma geométrica y ángulos del triángulo del brazo
*norm_4_P = sqrt(pow(aux_z, 2)+pow(aux_x, 2))* #<----- Calcula la distancia euclidiana total (norma) desde el hombro hasta el centro de la muñeca.

*epsilon = acos(aux_z/norm_4_P)* #<----- Calcula el ángulo auxiliar $\epsilon$ formado entre el vector de posición de la muñeca y el eje vertical.

*alpha = acos((pow(L_1, 2)+pow(norm_4_P, 2)-pow(L_2, 2))/(2*L_1*norm_4_P))** #<----- Aplica la Ley de Cosenos para determinar el ángulo interno $\alpha$ del primer eslabón móvil.

### Resolución de los ángulos de articulación ($\theta_2, \theta_3, \theta_4, \theta_5$)
*theta_2 = epsilon - alpha* #<----- Halla el ángulo cinemático de la segunda articulación $\theta_2$ mediante diferencia geométrica.

*theta_3 = 3.1416 - asin((sin(alpha)*sqrt(pow(aux_x, 2) + pow(aux_z, 2)))/(L_2))* #<----- Aplica la Ley de Senos para calcular la flexión del codo correspondiente al ángulo $\theta_3$.

*theta_4 = theta_1_P - theta_2 - theta_3* #<----- Deducción matemática para obtener el ángulo de orientación de la muñeca $\theta_4$.

*theta_5 = theta_g* #<----- Asigna directamente el ángulo de giro (roll) de la herramienta a la quinta articulación $\theta_5$.

### Retorno de las soluciones en formato vectorial compatible con ROS 2
*return [ float(theta_1), float(-theta_2), float(-theta_3), float(-theta_4), float(theta_5)]* #<----- Devuelve los 5 ángulos articulares calculados como una lista de valores flotantes aplicando inversiones de signo según los ejes del robot.


### --- Función Externa para el Estado de la Pinza (`gripper_state`) ---

### Declaración de la función y definición del parámetro escalar de entrada
**def gripper_state(theta):** #<----- Define la función auxiliar encargada de mapear el valor de apertura o cierre único al mecanismo multiarticular de la pinza.

### Retorno de la configuración simétrica para los 6 eslabones mecánicos del gripper
**return [float(-theta), float(theta), float(-theta), float(theta), float(-theta), float(theta)]** #<----- Distribuye el ángulo $\theta$ en una lista de valores flotantes alternando signos para controlar simétricamente la cinemática de los dedos mecánicos derecho e izquierdo de la pinza.




### --- Función Principal del Ciclo de Vida del Nodo (`main`) ---

### Declaración de la función principal y recepción de argumentos del sistema
**def main(args=None):** #<----- Define la función principal encargada de inicializar y coordinar la ejecución del nodo de control en ROS 2.

### Inicialización del middleware de comunicación de ROS 2
**rclpy.init(args=args)** #<----- Inicializa la capa de abstracción y los canales de comunicación de ROS 2 para Python.

### Instanciación del nodo de control del robot Dofbot
**node = DofbotControlNode()** #<----- Crea una instancia de nuestra clase controladora activando sus publicadores y el temporizador periódico.

### Bloqueo activo del script para la escucha continua de eventos y callbacks
**rclpy.spin(node)** #<----- Mantiene el nodo vivo en un bucle infinito procesando de forma constante las rutinas de movimiento del temporizador.

### Cierre controlado de los canales de comunicación y liberación de recursos
**rclpy.shutdown()** #<----- Finaliza limpiamente el entorno de ROS 2 y desconecta el nodo de la red al detener el programa.


### --- Punto de Entrada Oficial del Ejecutable de Python ---

### Validación del contexto de ejecución directa del archivo script
**if __name__ == "__main__":** #<----- Estructura condicional estándar que verifica si el script está siendo lanzado directamente desde la terminal.

### Activación del flujo completo del programa
**main()** #<----- Llama a la función principal para arrancar todo el sistema secuencial de control del brazo robótico.


## Funciones Principales del Código

### 1. Gestión de Tópicos Independientes para Brazo y Herramienta
El nodo coordina dos flujos de comunicación paralelos mediante publicadores de ROS 2:
*   Maneja las trayectorias del brazo de 5 ejes a través del tópico `/dofbot_trajectory_controller/joint_trajectory`.
*   Controla mecánicamente los 6 eslabones de la pinza por medio del tópico `/dofbot_gripper_controller/joint_trajectory`.

### 2. Máquina de Estados Basada en Temporizador ($ \lambda $)
Utiliza un temporizador periódico que evalúa el estado actual de la rutina cada 0.5 segundos empleando una variable de control ($ \lambda $). Esta variable actúa como un secuenciador de pasos que avanza de forma automática una vez concluida cada etapa:
*   **$ \lambda = 0 $:** Apertura total de la pinza para aproximación.
*   **$ \lambda = 1 $:** Cierre completo de la pinza (fase de prueba/calibración).
*   **$ \lambda = 2 $:** Reapertura de preparación.
*   **$ \lambda = 3, 5, 6, 7 $:** Posicionamiento espacial del brazo en cuatro posturas cartesianas distintas ($ X, Y, Z $).
*   **$ \lambda = 4, 8 $:** Acciones intermedias de sujeción (Close) y liberación (Open) de la carga.
*   **$ \lambda = 9 $:** Secuencia de seguridad que retorna todo el brazo a su postura de origen (`Home`).

### 3. Solucionador de Cinemática Inversa Planar-Vertical (`dofbot_ink`)
Transforma las coordenadas cartesianas tridimensionales de destino ($ X_P, Y_P, Z_P $) y las orientaciones del efector final ($ \theta_{1\_P}, \theta_g $) ingresadas en cada estado, en los 5 ángulos físicos en radianes que necesitan los servomotores. Utiliza para ello la combinación de:
*   Funciones trigonométricas de dos parámetros (`atan2`) para la rotación de la base ($ \theta_1 $).
*   Desacoplamiento geométrico del eslabón de la muñeca ($ L_3 $).
*   La Ley de Cosenos y la Ley de Senos para deducir la flexión del hombro ($ \theta_2 $) y el codo ($ \theta_3 $).

### 4. Mapeo Articular Alternado para la Pinza (`gripper_state`)
Resuelve la cinemática de acoplamiento de la pinza. Convierte un único valor escalar de apertura deseada ($ \theta $) en un vector simétrico de 6 elementos flotantes. Alterna de forma automática los signos (`-theta` y `theta`) para sincronizar el movimiento de los engranajes opuestos que cierran y abren los dedos del gripper.

### 5. Sincronización Física y Control de Tiempos de Espera
Implementa retardos estratégicos en el hilo de ejecución principal empleando comandos `time.sleep()`. Configura esperas de **2 segundos** en el mensaje ROS 2 para suavizar los perfiles de velocidad del hardware, complementadas con pausas de hasta **15 segundos** en el script para garantizar que el brazo robótico alcance físicamente sus coordenadas antes de calcular el paso sucesivo.


## Análisis Comparativo de los Nodos de Control Robótico

### 1. Configuración del Modelo Mecánico y Grados de Libertad (DoF)

### Estructura cinemática del robot SCARA
**Primer Código (SCARA):** Diseñado para un robot de 3 grados de libertad (planares y lineales). Controla una estructura simplificada donde la orientación y la posición se resuelven en un plano $X-Y$ bidimensional.

### Estructura cinemática del robot Dofbot
**Segundo Código (Dofbot):** Diseñado para un brazo articulado de 5 grados de libertad en el espacio tridimensional ($X, Y, Z$), sumando además un actuador final complejo compuesto por 6 eslabones mecánicos independientes para la pinza.


### 2. Estrategia y Control de Trayectorias

### Generación de perfiles de movimiento rectilíneo continuo
**Primer Código (SCARA):** Implementa un control dinámico y continuo en línea recta. Calcula puntos cartesianos intermedios mediante una interpolación paso a paso en cada activación del temporizador para trazar una recta perfecta entre $P_1$ y $P_2$.

### Generación de perfiles secuenciales punto a punto
**Segundo Código (Dofbot):** Implementa un control punto a punto o por máquina de estados secuencial. El temporizador evalúa estados discretos ($\lambda = 0, 1, 2...$) donde el brazo salta a coordenadas espaciales fijas o realiza acciones de apertura/cierre de pinza aisladas, en lugar de interpolar trayectorias continuas entre ellas.


### 3. Complejidad Cinemática y Resolución Matemática

### Análisis geométrico en el espacio bidimensional
**Primer Código (SCARA):** Resuelve ecuaciones analíticas enlazadas dentro del mismo script mediante la Ley de Cosenos clásica, desacoplando únicamente el último eslabón lineal para hallar las variables articulares ($\theta_1, \theta_2, \theta_3$).

### Análisis espacial y desacoplamiento tridimensional
**Segundo Código (Dofbot):** Incrementa la complejidad matemática al requerir un desacoplamiento espacial completo en 3D. Calcula la rotación de la base en azimuth con `atan2`, proyecta vectores planos-verticales combinando la altura del hombro ($z_{0\_1}$), y aplica simultáneamente la Ley de Cosenos y la Ley de Senos para deducir el comportamiento tridimensional del brazo.


### 4. Arquitectura de Comunicación en ROS 2

### Manejo unificado de actuadores mediante un único tópico
**Primer Código (SCARA):** Cuenta con un único publicador dirigido al tópico `/scara_trajectory_controller/joint_trajectory` para controlar de manera centralizada todos los eslabones cinemáticos del robot en un solo mensaje.

### Manejo independiente y descentralizado de actuadores
**Segundo Código (Dofbot):** Emplea una arquitectura de comunicación dual y descentralizada. Utiliza dos publicadores independientes que apuntan a tópicos de control de trayectorias separados (`/dofbot_trajectory_controller/...` y `/dofbot_gripper_controller/...`) para independizar el comportamiento motriz del brazo respecto a las señales mecánicas de la pinza.


### 5. Sincronización Temporal y Estabilización Física

### Control de baja latencia con pausas constantes
**Primer Código (SCARA):** Configura el temporizador a una frecuencia de 1 Hz (cada 1.0 segundo). Usa pausas breves de `time.sleep(2)` para dar un margen de estabilización física pequeño durante el avance regular de la trayectoria rectilínea.

### Control por máquina de estados con retardos críticos de hardware
**Segundo Código (Dofbot):** Configura el temporizador a una frecuencia de 2 Hz (cada 0.5 segundos) para monitorear constantemente la máquina de estados. Sin embargo, debido a las inercias del brazo tridimensional, introduce retardos críticos de hardware (`Duration(sec=2)`) junto con pausas extensas en el código de hasta 10 y 15 segundos para garantizar que los servomotores alcancen la postura deseada antes de ejecutar la siguiente acción de sujeción o traslado.
